# Model comparison

Final checkpoints: EnvPG with its learned layout, IPPO and MAPPO with the reference layout. Scores below use the same 25 wind bins, 150 controlled steps per bin, deterministic actions and load coefficient 0.1. Training budgets differ; this is a checkpoint comparison. Training losses are diagnostics, not a shared performance measure.


In [ ]:
from pathlib import Path
import json, os, sys
ROOT = Path.cwd().resolve()
assert (ROOT / "envpg.py").exists(), "Open this notebook from the repository root"
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / "__simul__" / "matplotlib"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / "wfcrl-benchmark" / "algos"))
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
from envpg import Actor, REFERENCE, windrose
from wfcrl import environments as envs
from extractors import VectorExtractor
from baseline_ippo import Agent as IPPOAgent
from baseline_mappo import Agent as MAPPOAgent
from floris.tools.visualization import visualize_cut_plane, plot_turbines_with_fi

RUNS = {
    "EnvPG": ROOT / "runs/envpg-001",
    "IPPO": ROOT / "wfcrl-benchmark/runs/Dec_Ablaincourt_Floris__baseline_ippo__windrose_1__1794149416",
    "MAPPO": ROOT / "wfcrl-benchmark/runs/Dec_Ablaincourt_Floris__baseline_mappo__windrose_1__1794150208",
}
for name, path in RUNS.items():
    assert path.is_dir(), f"Missing {name} run: {path}"
config = json.loads((RUNS["EnvPG"] / "config.json").read_text())["parameters"]
H = config["horizon"]
for name in ("IPPO", "MAPPO"):
    baseline = yaml.load((RUNS[name] / "config.yaml").read_text(), Loader=yaml.BaseLoader)
    assert baseline["env_id"] == "Dec_Ablaincourt_Floris"
    assert baseline["scenario"] == "windrose"
    assert int(baseline["episode_length"]) == H + 1
    assert float(baseline["load_coef"]) == config["load_coef"]
print(f"{H} steps/bin; load coefficient {config['load_coef']}; runs: " + ", ".join(RUNS))


## Training

EnvPG reports mean reward per controlled step and losses per design iteration. WFCRL reports episode reward and PPO losses per environment step; its episode reward is divided by 150 below. The loss scales are not comparable across algorithms.


In [ ]:
events = EventAccumulator(str(RUNS["EnvPG"] / "tb")); events.Reload()
def scalar(tag):
    return pd.DataFrame([(x.step, x.value) for x in events.Scalars(tag)], columns=["step", "value"])

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
e = scalar("train/reward_mean")
ax[0].plot(e.step, e.value, label="EnvPG")
ax[0].set(xlabel="Design iteration", ylabel="Mean reward / step", title="EnvPG training")
for name in ("IPPO", "MAPPO"):
    farm = pd.read_csv(RUNS[name] / "farm.csv")
    ax[1].plot(farm["index"], farm["episode_reward"] / H, label=name, alpha=.8)
ax[1].set(xlabel="Environment steps", ylabel="Episode reward / 150", title="WFCRL training")
for a in ax: a.legend(); a.grid(alpha=.2)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
for tag, label in (("loss/actor", "Actor"), ("loss/critic", "Critic"), ("loss/cost_sensitivity", "Cost sensitivity")):
    e = scalar(tag); ax[0].plot(e.step, e.value, label=label)
ax[0].set(title="EnvPG losses", xlabel="Design iteration")
for a, name in zip(ax[1:], ("IPPO", "MAPPO")):
    losses = pd.read_csv(RUNS[name] / "losses.csv")
    for suffix, label in (("policy_loss", "Policy"), ("value_loss", "Value")):
        cols = [c for c in losses if c.endswith(suffix)]
        if cols: a.plot(losses["index"], losses[cols].mean(axis=1), label=label)
    a.set(title=f"{name} losses", xlabel="Environment steps")
for a in ax: a.legend(); a.grid(alpha=.2)
plt.tight_layout(); plt.show()


## Final checkpoints

All policies are rolled out in the WFCRL multi-agent FLORIS environment. Reward is summed over each wind bin; power and load proxy are averaged over steps. Wind-bin scores are weighted by SMARTEOLE frequencies.


In [ ]:
def load_model(name):
    env = envs.make("Dec_Ablaincourt_Floris", controls=["yaw"], log=False,
                    max_num_steps=H + 1, load_coef=config["load_coef"])
    if name == "EnvPG":
        # This checkpoint contains a NumPy layout; load only local, trusted runs.
        state = torch.load(RUNS[name] / "latest.pt", map_location="cpu", weights_only=False)
        actor = Actor(H, config["max_displacement"]).eval()
        actor.load_state_dict(state["actor"])
        layout = REFERENCE.copy(); layout[1:] += np.asarray(state["layout"]).reshape(6, 2)
        env.mdp.interface.fi.reinitialize(layout_x=layout[:, 0], layout_y=layout[:, 1])
        return {"env": env, "actor": actor, "layout": layout, "z": np.asarray(state["layout"])}
    cls = IPPOAgent if name == "IPPO" else MAPPOAgent
    first = env.possible_agents[0]
    obs = VectorExtractor(env.observation_space(first))
    act = VectorExtractor(env.action_space(first))
    baseline = yaml.load((RUNS[name] / "config.yaml").read_text(), Loader=yaml.BaseLoader)
    layers = tuple(map(int, baseline["hidden_layer_nn"]))
    agents = []
    for i in range(env.num_turbines):
        agent = cls(obs.space, act.space, layers).eval()
        path = RUNS[name] / f"baseline_{name.lower()}.cleanrl_model_{i}"
        agent.load_state_dict(torch.load(path, map_location="cpu", weights_only=True))
        agents.append(agent)
    return {"env": env, "agents": agents, "obs": obs, "act": act, "layout": REFERENCE.copy()}

models = {name: load_model(name) for name in RUNS}

def run_case(name, direction, speed):
    m = models[name]; env = m["env"]
    env.reset(options={"wind_speed": float(speed), "wind_direction": float(direction)})
    first = env.possible_agents[0]
    rows = []
    with torch.no_grad():
        for _ in range(H):
            if name == "EnvPG":
                usage = [env.accumulated_actions[a]["yaw"] for a in env.possible_agents]
                s = np.r_[env._state["freewind_measurements"], env._state["yaw"], usage, env.num_moves]
                action = m["actor"].sample(torch.tensor(s, dtype=torch.float32)[None],
                                           torch.tensor(m["z"], dtype=torch.float32)[None],
                                           deterministic=True)[0].numpy()
            for i, turbine in enumerate(env.possible_agents):
                if name == "EnvPG":
                    command = {"yaw": np.array([action[i]], dtype=np.float32)}
                else:
                    obs = torch.tensor(m["obs"](env.observe(turbine)), dtype=torch.float32)
                    agent = m["agents"][i]
                    if name == "IPPO":
                        action_i = agent.get_action_and_value(obs, deterministic=True)[0]
                    else:
                        action_i = agent.get_action(obs, deterministic=True)[0]
                    command = m["act"].make_dict(action_i)
                env.step(command)
            rows.append({"reward": float(env.rewards[first].item()),
                         "power_mw": sum(env.infos[a]["power"] for a in env.possible_agents),
                         "load_proxy": np.mean(np.abs([env.infos[a]["load"] for a in env.possible_agents])),
                         "yaw": np.asarray(env._state["yaw"]).copy()})
    return rows


In [ ]:
conditions = windrose(config["wind_data"])
assert len(conditions) == 25 and abs(sum(w for _, _, w in conditions) - 1) < 1e-9
scores, by_bin = [], []
for name in RUNS:
    for direction, speed, weight in conditions:
        rows = run_case(name, direction, speed)
        reward = sum(r["reward"] for r in rows)
        by_bin.append({"model": name, "direction": direction, "speed": speed,
                       "weight": weight, "reward": reward})
        scores.append({"model": name, "weight": weight, "reward": reward,
                       "power_mw": np.mean([r["power_mw"] for r in rows]),
                       "load_proxy": np.mean([r["load_proxy"] for r in rows])})
frame = pd.DataFrame(scores)
summary = pd.DataFrame({key: (frame[key] * frame.weight).groupby(frame.model, sort=False).sum()
                        for key in ("reward", "power_mw", "load_proxy")})
display(summary.round(3))
fig, ax = plt.subplots(1, 3, figsize=(11, 3))
for a, key, title in zip(ax, summary.columns, ("Weighted reward / 150 steps", "Farm power (MW)", "Load proxy")):
    a.bar(summary.index, summary[key]); a.set(title=title); a.grid(axis="y", alpha=.2)
plt.tight_layout(); plt.show()


## One wind condition

The traces and hub-height wakes below use 8 m/s wind from 270° after 150 control steps. This is a visual example; the weighted results above cover the full windrose.


In [ ]:
example = {name: run_case(name, 270.0, 8.0) for name in RUNS}
fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
for name, rows in example.items():
    for a, key in zip(ax, ("reward", "power_mw", "load_proxy")):
        a.plot([r[key] for r in rows], label=name)
for a, title in zip(ax, ("Reward / step", "Farm power (MW)", "Load proxy")):
    a.set(title=title, xlabel="Control step"); a.grid(alpha=.2)
ax[0].legend(); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(13, 3))
for a, (name, rows) in zip(ax, example.items()):
    yaw = np.stack([r["yaw"] for r in rows]).T
    im = a.imshow(yaw, aspect="auto", origin="lower", cmap="coolwarm", vmin=-40, vmax=40)
    a.set(title=f"{name} yaw (°)", xlabel="Control step", ylabel="Turbine")
    a.set_yticks(range(7), range(1, 8))
fig.colorbar(im, ax=ax, label="Yaw (°)"); plt.show()


In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(11, 10), sharex=True, sharey=True)
for a, name in zip(ax, RUNS):
    fi = models[name]["env"].mdp.interface.fi
    yaw = fi.floris.farm.yaw_angles.copy()
    plane = fi.calculate_horizontal_plane(height=90.0, x_resolution=100, y_resolution=50,
                                          x_bounds=(200, 2800), y_bounds=(-600, 600),
                                          wd=[270.0], ws=[8.0], yaw_angles=yaw)
    image = visualize_cut_plane(plane, ax=a, min_speed=0, max_speed=8,
                                levels=np.linspace(0, 8, 21))
    plot_turbines_with_fi(fi, ax=a, wd=np.array([270.0]), yaw_angles=yaw)
    for i, (x, y) in enumerate(models[name]["layout"]):
        a.text(x, y + 65, str(i + 1), ha="center", fontsize=8)
    a.set(title=name, ylabel="y (m)")
ax[-1].set_xlabel("x (m)")
fig.colorbar(image, ax=ax, label="Wind speed (m/s)")
plt.show()
for m in models.values(): m["env"].close()
